# Simple Agentic RAG

This notebook walks through building an Agentic RAG system using LangChain and FAISS.

**Steps:**
1. Install dependencies
2. Ingest documents into a FAISS vector store
3. Build a LangChain agent with a document search tool
4. Ask questions

## 1. Install Dependencies

In [1]:
%pip install -r  requirements.txt -q


Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from types import ModuleType

# Stub out optional langchain_community providers that aren't installed,
# so importing the package doesn't raise ModuleNotFoundError at load time.
for _mod in [
    "langchain_community.chat_models.vertexai",
    "langchain_community.chat_models.google_palm",
    "langchain_community.llms.vertexai",
]:
    if _mod not in sys.modules:
        sys.modules[_mod] = ModuleType(_mod)

## 2. Set API Keys

In [3]:
import os

from dotenv import load_dotenv


#os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN
load_dotenv(os.path.expanduser("~/FDP-AGENENTIC-AI-RAG/.env"))
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")


# huggingface_hub caches HF_HUB_OFFLINE at import time — reset it directly
#import huggingface_hub.constants as _hf_const
#_hf_const.HF_HUB_OFFLINE = False

## 3. Ingest Documents into FAISS Vector Store

Defines knowledge inline, splits it into chunks, embeds them, and saves the vector store locally.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

fictitious_department_info = [
    "The Department of Intelligent Systems Engineering (DISE) is a small, focused department that works on applied AI and intelligent systems.",
    "It currently has around 40 students, with a healthy mix of undergraduate and postgraduate learners.",
    "The department is run by a team of 14 professors, including experienced faculty members and a few industry practitioners.",
    "Students can choose from about 5 courses, ranging from core subjects to electives and hands-on project work.",
    "Overall, DISE aims to prepare students for real-world engineering roles through practical learning and industry exposure.",
]

documents = [Document(page_content=text) for text in fictitious_department_info]

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
docs = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2",
    model_kwargs={"local_files_only": True},
)
db = FAISS.from_documents(docs, embeddings)
db.save_local("vectorstore")

print(f"Vector DB created successfully ({len(docs)} chunks)")

/tmp/ipykernel_77558/2126607911.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector DB created successfully (5 chunks)


## 4. Build the RAG Agent

Loads the vector store and wires it up as a tool for a LangChain ZERO_SHOT_REACT agent.

In [5]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2",
    model_kwargs={"local_files_only": True},
)

db = FAISS.load_local(
    "vectorstore",
    embeddings,
    allow_dangerous_deserialization=True,
)

@tool
def search_answers(query: str) -> str:
    """Search the knowledge base"""
    results = db.similarity_search(query, k=2)
    return "\n".join([doc.page_content for doc in results])

# ── Switch providers by setting MODEL_NAME — no code changes needed ──────────
#   google_genai → MODEL_NAME=google_genai:gemini-2.5-flash   (default)
#   openai       → MODEL_NAME=openai:gpt-4o
#   anthropic    → MODEL_NAME=anthropic:claude-3-5-sonnet-20241022
#   huggingface  → MODEL_NAME=huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0
#   ollama       → MODEL_NAME=ollama:llama3
# ────────────────────────────────────────────────────────────────────────────

# Make loaded keys visible to init_chat_model via standard env-var names
os.environ.setdefault("GOOGLE_API_KEY", GEMINI_API_KEY or "")

MODEL_NAME = os.getenv("MODEL_NAME", "google_genai:gemini-2.5-flash")
chat_llm = init_chat_model(MODEL_NAME, temperature=0.1)

agent = create_agent(
    model=chat_llm,
    tools=[search_answers],
    system_prompt=(
        "You are an intelligent Agentic RAG system. "
        "Decide whether document retrieval is needed. "
        "If needed, use the search_answers tool."
    ),
)

print(f"Agent ready (model: {MODEL_NAME}).")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Agent ready (model: google_genai:gemini-2.5-flash).


## 5. Ask Questions

Edit `question` below and re-run this cell to query the agent.

In [6]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print("\nAnswer:", result["messages"][-1].content)


ask("What is DISE?")


Answer: [{'type': 'text', 'text': 'The Department of Intelligent Systems Engineering (DISE) is a small, focused department that works on applied AI and intelligent systems. Its main goal is to prepare students for real-world engineering roles through practical learning and industry exposure.', 'extras': {'signature': 'CskDAQw51sf7LMJi/3VlIJ6o8tc8x7QWc7thU9+clapaOC9y0h/P7lnWMtBGErQFK0T3SCpnAdBeuKBrzW1U7VP+ECxbeuOBwtEZ5apbfSKfko86zyFIPoI/87ZWnOrCe9K0lMmR2hdWJq15phqpBIGKX7rTCXADyMEb+7iYCD9EBcTZvp6TlmiI8oJk5B0V3Xf82gMDl+I4h7miyIZQVd822PgofEVYQ+CKso2yTUC/tT7C2aMADXBlRzEZRIsp46L7RmlWwBw2ZICIkctry4pW+KddMjqoH4iPCO9fx7+l0diXvHjtYbf15Fygt3nIEekZqixkSvpt8BrJ6XsqYg0cxqDc9lkg1UPkdg1yS9YnEkdbUXq8oyxxR3YM1hntIng/peVZNhc7XbFpPTQBX0be4StqVb9FpNQBe+k/dHYKg2wUgaHK51NuJLa6GBfF543hsfYchb7G+w2+r9UOUmbFkULKIxQO5TJDgFk7xqKu47Yfal7xOIwfPo9RmDiweHdeRS1Bx0joyuVsHqwPhGEbmgN+5r+BW5vw2yyGX1vsNwepG4OOuV8YleR5CZ8cvkRIpjcXuLNxklPj/Q7S/YwO/KUmUUuMuIHbnw=='}}]


In [7]:
ask("How many students does DISE have?")


Answer: [{'type': 'text', 'text': 'DISE currently has around 40 students, including both undergraduate and postgraduate learners.', 'extras': {'signature': 'Ct8CAQw51sdcE4qX8Ak6Yp8q+eVWtMEO4DExgbRW7vo2Ic0f5AIbYeoF8DJMkCUZoFHib/T5iFu2pPCFr+9UcreOor+FY5qTP+JPEUTg7Us5VxsuQkEFLbZRPf/Hg+3/FIC2g6rErqxfDqgofUXBr57u8csF0iG+pv8RuP6RpSUwDt43guj4vn16M9DG4Q/AO/5G9TKzqncvw3IWD2r+CV/gVqpZ1UQE72gHVO19pbkKIzDrsECDnxZpCLanxhXpt/uVbE8lwAKYG6BC/p6XcSBn3YjzHwNEGQ0adoaQtLI5M51/0wnoxsptDtuPFSUft6dBPLdLp7L/MrJ3pu/oos9ogzrpWdjqYpK2BJ0Y9BAeNbgqku8faU7ODjRDIqcWn6vazSQs57xEzrkea4wgSpVYbxbQWCYZ1hEwpmTAsNx8UoLKrsset9+7e+lI2GXzNLd+D40XC0E0F2chRB+TpLHJ'}}]


In [8]:
ask("How many stuents does it have")


Answer: [{'type': 'text', 'text': 'It currently has around 40 students, with a healthy mix of undergraduate and postgraduate learners.', 'extras': {'signature': 'Cr8DAQw51sdpGiHr2Vy0hEdK1P36YLKRC8XBrlYM9rtStyfvVIq+zYUoCwrfBwYfd9nt/t1mmAQITqICceby0xhJac3ON3vEgU+s/9jf1nURHjewSlqCudKefOe2EusTwvbTEE3ceUFrbArUAGJsc/uKPCHdshwqqtgc5jOpz4NviunY9UlG3JP/Uze11pbfgaC2N72Wkq7ER989ybrd+KG+N6UHH3o0ea4UU/orFaxGNqhe+QPFZyiDpR1JDusECL1MSXe8GFiijlqvjwQ2nwDe2sKaGIzOlLZqHNmpj77OD57bckUwLNDf/Utl0ztKKJXqXAGQGpo61a0ogd/v4uKUOEoY4RkWMGV4JRbnBb8SyIWktzjzcE5UkQ8/aztdt97vtyuool3kvzShqrkiNOckoCZctIQVEQASQYBCN5M1U1nAjGyu3/cfo+Zj/N9CA1/wVrFOhvsthHqod6i5mnPEA+farfdl3cIeUjzkN5BiWBS1AS/10RlFnaN8OWZ43x6PswutjjFRcaDuHjL6JVTulRMsXfjq/2PI1EYR81DFK5FUj7IaAThqXTmNopMArAqBfcjPRG8yRv3Mu1W/7cZd'}}]


## 6. Evaluate with RAGAS

[RAGAS](https://docs.ragas.io/) measures RAG pipeline quality across four metrics:

| Metric | What it measures | Needs ground truth? |
|---|---|---|
| **Faithfulness** | Is the answer grounded in the retrieved context? | No |
| **Answer Relevancy** | Is the answer relevant to the question? | No |
| **Context Precision** | Are the retrieved chunks actually useful? | Yes |
| **Context Recall** | Did retrieval capture what was needed to answer? | Yes |

Scores range from 0 to 1 — higher is better.

In [9]:
%pip install "ragas>=0.2" -q

Note: you may need to restart the kernel to use updated packages.


In [10]:
def extract_answer(agent_response: dict) -> str:
    """Pull the plain-text answer out of the agent's message (handles list or str content)."""
    content = agent_response["messages"][-1].content
    if isinstance(content, list):
        return " ".join(
            item.get("text", "") for item in content if isinstance(item, dict)
        ).strip()
    return str(content).strip()


# --- Ground-truth answers (reference) ---
eval_questions = [
    "What is DISE?",
    "How many students does DISE have?",
    "How many professors are in DISE?",
    "What courses are offered in DISE?",
    "What is the goal of DISE?",
]

ground_truths = [
    "The Department of Intelligent Systems Engineering (DISE) is a small, focused department that works on applied AI and intelligent systems.",
    "DISE currently has around 40 students, with a healthy mix of undergraduate and postgraduate learners.",
    "The department is run by a team of 14 professors.",
    "Students can choose from about 5 courses, ranging from core subjects to electives and hands-on project work.",
    "DISE aims to prepare students for real-world engineering roles through practical learning and industry exposure.",
]

# --- Collect RAG answers and retrieved contexts ---
answers, contexts = [], []

for question in eval_questions:
    # Retrieve chunks used as context
    retrieved = db.similarity_search(question, k=2)
    contexts.append([doc.page_content for doc in retrieved])

    # Get the agent's answer
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    answers.append(extract_answer(response))

# Preview
for q, a, ctx in zip(eval_questions, answers, contexts):
    print(f"Q: {q}")
    print(f"A: {a}")
    print(f"Ctx[0]: {ctx[0][:80]}...")
    print()

Q: What is DISE?
A: The Department of Intelligent Systems Engineering (DISE) is a small, focused department that specializes in applied AI and intelligent systems. Its main goal is to prepare students for real-world engineering roles through practical learning and industry exposure.
Ctx[0]: The Department of Intelligent Systems Engineering (DISE) is a small, focused dep...

Q: How many students does DISE have?
A: DISE currently has around 40 students, including both undergraduate and postgraduate learners.
Ctx[0]: It currently has around 40 students, with a healthy mix of undergraduate and pos...

Q: How many professors are in DISE?
A: There are 14 professors in DISE.
Ctx[0]: The department is run by a team of 14 professors, including experienced faculty ...

Q: What courses are offered in DISE?
A: I couldn't find a list of specific courses offered in DISE. The Department of Intelligent Systems Engineering (DISE) focuses on applied AI and intelligent systems, aiming to prepare students

In [12]:
import sys
from types import ModuleType

# ragas tries to import ChatVertexAI from the stub — add a dummy class to satisfy it
for _mod_name, _attrs in {
    "langchain_community.chat_models.vertexai": ["ChatVertexAI"],
    "langchain_community.chat_models.google_palm": ["ChatGooglePalm"],
    "langchain_community.llms.vertexai": ["VertexAI"],
}.items():
    _mod = sys.modules.setdefault(_mod_name, ModuleType(_mod_name))
    for _attr in _attrs:
        if not hasattr(_mod, _attr):
            setattr(_mod, _attr, type(_attr, (), {}))

from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

judge_llm = LangchainLLMWrapper(chat_llm)
judge_embeddings = LangchainEmbeddingsWrapper(embeddings)

samples = [
    SingleTurnSample(
        user_input=q,
        response=a,
        retrieved_contexts=ctx,
        reference=gt,
    )
    for q, a, ctx, gt in zip(eval_questions, answers, contexts, ground_truths)
]
ragas_dataset = EvaluationDataset(samples=samples)

result = evaluate(
    dataset=ragas_dataset,
    metrics=[
        Faithfulness(llm=judge_llm),
        AnswerRelevancy(llm=judge_llm, embeddings=judge_embeddings),
        ContextPrecision(llm=judge_llm),
        ContextRecall(llm=judge_llm),
    ],
)

import pandas as pd

scores_df = result.to_pandas()[
    ["user_input", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]
]
scores_df.loc["mean"] = scores_df.mean(numeric_only=True)
scores_df.iloc[-1, 0] = "MEAN"

pd.set_option("display.float_format", "{:.3f}".format)
print(scores_df.to_string(index=False))

/tmp/ipykernel_77558/3602951817.py:16: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/tmp/ipykernel_77558/3602951817.py:16: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/tmp/ipykernel_77558/3602951817.py:16: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Faithfulness, 

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


                       user_input  faithfulness  answer_relevancy  context_precision  context_recall
                    What is DISE?         1.000             0.629              1.000           1.000
How many students does DISE have?         1.000             0.975              1.000           1.000
 How many professors are in DISE?         0.000             0.995              1.000           1.000
What courses are offered in DISE?         0.667             0.000              1.000           1.000
        What is the goal of DISE?         1.000             0.679              0.500           1.000
                             MEAN         0.733             0.656              0.900           1.000


## 7. Interpreting the RAGAS Results

### What each metric tells you

| Metric | Question it answers | Score guide |
|---|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved chunks? | < 0.5 → hallucination risk; > 0.8 → trustworthy |
| **Answer Relevancy** | Does the answer actually address the question asked? | < 0.5 → off-topic; > 0.8 → on-point |
| **Context Precision** | Are the top-ranked retrieved chunks the useful ones? | < 0.5 → noisy retrieval; > 0.8 → precise retrieval |
| **Context Recall** | Did retrieval surface everything needed to answer? | < 0.5 → missing evidence; > 0.8 → good coverage |

---

### Reading the scores in this pipeline

**Faithfulness** measures whether the model "sticks to the script." A small model like TinyLlama-1.1B tends to score lower here because it may add plausible-sounding details not present in the retrieved text (hallucination). A cloud model like Gemini typically scores higher because it follows instructions more precisely.

**Answer Relevancy** reflects how well the question is understood. Short, focused answers on a narrow knowledge base (like DISE department info) usually score well. If the agent wanders off-topic or refuses to answer, this drops.

**Context Precision & Recall** are about retrieval quality, not the LLM. Since we use a small FAISS store (5 chunks) with sentence-transformer embeddings, precision is expected to be high — there is little noise to retrieve. Recall may suffer on multi-fact questions if the relevant chunk wasn't ranked in the top-k.

---

### What to do if scores are low

| Symptom | Likely cause | Fix |
|---|---|---|
| Low Faithfulness | LLM adds facts not in context | Use a stronger/instruction-tuned model; tighten the system prompt |
| Low Answer Relevancy | Model misunderstands the question | Improve the system prompt; switch to a better model |
| Low Context Precision | Wrong chunks retrieved | Tune `k`, try MMR retrieval, or re-embed with a better model |
| Low Context Recall | Needed chunk not retrieved | Increase `k`, reduce chunk size, or improve the embedding model |

---

> **Note on the judge LLM:** RAGAS uses the same `judge_llm` to score faithfulness and relevancy. If the judge is also TinyLlama (a small model), scores may be noisy or inconsistent — a stronger model (Gemini, GPT-4o) as the judge gives more reliable scores even when TinyLlama is the RAG agent.